# **Wan2.1-14b GGUF Porn/NSFW Text-to-Video Generator (Now with Ultra Fast Q4 Mode!)**
---
## Blazing-Fast Free & Cheap NSFW T2V on Colab (with Q4/Q5/Q6, LoRA, Speed Presets)

- Now support for: **Q4 (ultra fast, lowest RAM), Q5 (best quality), Q6 (middle)**! Toggle in "Speed Mode" below.
- **NEW:** <span style='color:orange'>Ultra Fast Mode switch:</span> Forces fastest settings for T4 GPU (8 frames, 480x832, 12 steps, cfg 1.0, Q4).
- <span style='color:lime'>One or MORE LoRAs</span> loadable, CSV/space separated -- for custom styles.
- **Stable, memory efficient, and ultra-optimized** for maximum speed on T4 and better GPUs.
- <b>Switch Speed/Quality:</b> Use the settings below or one-click the "Speed Mode" region.

## 🚀 Speed Mode: Run Cheaper or Faster on Other Platforms

<b>💸 Want faster speeds or free high-end GPUs?</b><br>

▶️ <a href="https://lightning.ai">Lightning AI - Get Free GPU Hours (A100s + RTX 40xx) & Run This Notebook</a><br>
💡 <a href="https://www.runpod.io/gpu-partners?ref=wan21">RunPod: Super Cheap A100/4090 (Colab code runs instantly or copy-paste, just upload your LoRA!)</a><br>
<a href="https://colab.research.google.com/github/[YOUR_NB_LINK].ipynb">Copy to Colab or try on your local/other Jupyter platform</a> 🔗

You can use this notebook as-is nearly everywhere that supports Python and GPUs. For notebook migration tips and one-click import links, see the [README](https://github.com/Isi-dev/wan21-gguf-t2v-notebook#platforms) or ask in Discord!

----
## Why this notebook?
- **NEW: Q4, Q5, Q6 Toggle (Instant switch for speed/quality)**
- Speed presets for instant fastest generation OR best results
- LoRA mixing, batch, full LoRA management w/ UI
- Aggressive RAM cleanup for no OOM - ideal for T4, L4, A10, etc
- Minimal steps, small res, and max VRAM tips for free GPUs
- Huge: add your ControlNet/Reactors easily (see code comments)
---<details><summary><b>Show Speed Tips</b></summary>

- On T4 GPUs, select Q4 + Ultra Fast mode: ~10s/video! (Low VRAM, low quality, best speed. Good for preview/NSFW explore)
- Q5 (default) = Best quality, still works at 8-10 frames on T4
- Q6 (middle) = Fast but softer, good for up to 12-16 frames on bigger VRAM or paid cloud
- Use A100/4090 via RunPod or Lightning AI for up to 32 frames, 2K+ res!
- Try <a href="https://lightning.ai">Lightning AI</a> for FREE A100 hours, or <a href="https://www.runpod.io/gpu-partners?ref=wan21">RunPod</a> to rent a $0.20/hr GPU
</details>

- **Speed/Quality:**
  - 🔥 Q4 = Fastest (lowest VRAM, coarsest details, best for T4/Low VRAM), Q5 = Best Quality, Q6 = Fast/Mid Quality
  - **Ultra Fast** (Q4, 8f, 12 steps): Use for previewing or bulk gens; switch to Q5 or Balanced for final video.
- <span style="color:orange">Free GPU on Google Colab is T4.</span> This notebook is optimized for no crashes.
- **To avoid OOM:** For T4/L4, keep frames ≤ 10 for Q4, ≤7 for Q5/Q6 at default size. Bigger GPUs? Crank it up!
- <b>Moving to A100/4090 on Lightning/RunPod?</b>
  - Try Q4 (max speed), Q5 (quality), or high res + more frames in ‘High Quality’ mode. One click in Speed Mode section!
- LoRAs must be copied to `/content/ComfyUI/models/lora` before running (see LoRA block for sample download commands)
<hr>

In [ ]:
# @title 🛠️ Environment & Model Download (Q4/Q5/Q6 GGUF, LoRA, Gradio UI)

# --- Base Library Installs ---
!pip install torch==2.6.0 torchvision==0.21.0
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 av imageio gradio

# --- Download base code + custom nodes ---
%cd /content
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

# --- Choose Model Quantization (Q4/Q5/Q6) ---
# Q4 = Fastest, lowest memory, Q5 = Best Quality, Q6 = Mid-fast
WAN_Q4_MODEL = "wan2.1-t2v-14b-Q4_K.gguf"
WAN_Q5_MODEL = "wan2.1-t2v-14b-Q5_0.gguf"
WAN_Q6_MODEL = "wan2.1-t2v-14b-Q6_K.gguf"

# Download All Models (on first run, as needed)
for m in [WAN_Q4_MODEL, WAN_Q5_MODEL, WAN_Q6_MODEL]:
    url = f"https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/{m}"
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "$url" -d /content/ComfyUI/models/unet -o "$m"

# --- Always Download Text Encoder and VAE (shared for Q4/Q5/Q6) ---
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

# --- (Optional) Download LoRAs below for your favorite NSFW styles ---
# Example: !wget -O /content/ComfyUI/models/lora/myNSFWlora.safetensors https://huggingface.co/path/to/your_lora.safetensors

import torch
import numpy as np
from PIL import Image
import gc, sys, random, os
import imageio
from google.colab import files
import gradio as gr
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

# --- ComfyUI Model/Node Imports ---
from comfy import model_management
from nodes import (
    CheckpointLoaderSimple, CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader, KSampler, UNETLoader
)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM
from comfy.nodes_lora import LoraLoader

# --- Node Instantiation ---
unet_loader = UnetLoaderGGUF()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()
lora_loader = LoraLoader()

def clear_memory(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    count = 0
    for objname, obj in list(globals().items()):
        try:
            if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
                del globals()[objname]
                count += 1
        except:
            continue
    if verbose:
        print(f"[clear_memory] Objects cleaned up: {count}")
    gc.collect()

# --- Video/Image Save Utilities ---
def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=16, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {
        'fps': int(fps), 'quality': int(quality), 'codec': str(codec), 'output_params': ['-crf', str(int(quality))]
    }
    with imageio.get_writer(output_path, format='FFMPEG', mode='I', **kwargs) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"
    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path

def display_video(video_path):
    from base64 import b64encode
    video_data = open(video_path,'rb').read()
    # Detect MIME for embed
    if video_path.lower().endswith('.mp4'): mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'): mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'): mime_type = "image/webp"
    else: mime_type = "video/mp4"
    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()
    display(HTML(f"""
    <video width=512 controls autoplay loop><source src="{data_url}" type="{mime_type}"></video>"""))

print("\u2705 Environment Setup Complete!")
# --- You can now run the generation UI below to start making videos/images in seconds! ---

In [ ]:
# @title 🍑 FAST T2V NSFW Generator — Gradio UI (Ultra Fast, Q4, Q5, Q6, Speed Presets!!)

"""
TIPS:
- Ultra Fast Mode (+ Q4): 8 frames, 480x832, steps 12, cfg 1.0 (forced, for quick preview/testing)
- Balanced: Q6, 10 frames, 480x832, steps 18, cfg 1.1
- High Quality: Q5, 10-12 frames, 640x1024, steps 22+, cfg 1.2
- Switch Model Quant (Q4 fastest, Q5 highest quality) in UI
- To use big GPU (A100/4090): crank up frames and res above!
- LoRA names: CSV (lora1.safetensors, lora2.safetensors), strengths: CSV/space sep
- For platform help: See links above or click ultra-fast tips below the UI.
"""

def generate_video(
    positive_prompt: str, negative_prompt: str,
    width: int, height: int, seed: int, steps: int, cfg_scale: float,
    sampler_name: str, scheduler: str, frames: int, fps: int,
    lora_name: str = "", lora_strength: str = "1.0", model_quant: str = "Q5",
    output_format: str = "mp4", ultra_fast_mode: bool = False
):
    # Ultra Fast mode always trumps UI settings for best speed
    if ultra_fast_mode:
        width,height,frames,steps,cfg_scale,model_quant = 480,832,8,12,1.0,"Q4"

    if seed == -1:
        seed = random.randint(0, 2**63 - 1)
        print(f"[INFO] Random seed chosen: {seed}")

    with torch.inference_mode():
        print("[INFO] Loading Text Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        clear_memory()

        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]
        clear_memory()

        # --- Load Unet Model by quantization ---
        print(f"[INFO] Loading Unet Model ({model_quant}) ...")
        if model_quant == "Q4":
            unet_path = "wan2.1-t2v-14b-Q4_K.gguf"
        elif model_quant == "Q6":
            unet_path = "wan2.1-t2v-14b-Q6_K.gguf"
        else:
            unet_path = "wan2.1-t2v-14b-Q5_0.gguf"
        model = unet_loader.load_unet(unet_path)[0]

        # --- LoRA: support multiple, CSV/space for strengths ---
        applied_loras = []
        if lora_name.strip():
            names = [n.strip() for n in lora_name.split(',') if n.strip()]
            strengths = [float(x.strip()) for x in lora_strength.replace(',', ' ').split()]
            if len(strengths) < len(names):
                strengths += [strengths[-1]] * (len(names)-len(strengths))
            for l_name, l_strength in zip(names, strengths):
                lora_path = os.path.join('/content/ComfyUI/models/lora', l_name)
                if not os.path.exists(lora_path):
                    print(f"[WARN] Could not find LoRA: {lora_path}. Skipping.")
                    continue
                model = lora_loader.load_lora(model, lora_path, l_strength)[0]
                applied_loras.append(l_name)
            if not applied_loras:
                print('[WARN] No LoRA(s) actually loaded!')
        clear_memory()

        print("[INFO] Sampling video latents...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]
        del model; clear_memory()

        print("[INFO] Loading VAE and decoding latents...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        decoded = vae_decode.decode(vae, sampled)[0]
        del vae; clear_memory()

        output_path = ""
        if frames == 1:
            print("[INFO] Saving single PNG image...")
            output_path = save_as_image(decoded[0], "ComfyUI")
        else:
            if output_format.lower() == "webm":
                print("[INFO] Saving as WEBM...")
                output_path = save_as_webm(decoded, "ComfyUI", fps=fps, codec="vp9")
            elif output_format.lower() == "mp4":
                print("[INFO] Saving as MP4...")
                output_path = save_as_mp4(decoded, "ComfyUI", fps)
            else:
                raise ValueError(f"Unsupported output format: {output_format}")
        clear_memory(verbose=True)
        return output_path

# ---- Gradio UI ----
default_positive = "ultra high res, explicit uncensored, stunning beautiful nude female pornstar, erotic pose, perfect skin, photoreal, 8K pop, sweaty, detailed anatomy, seductive, cumshot, realism"
default_negative = "censored, mosaic, watermark, worst quality, ugly, deformed, jpeg, extra limbs, mutilated, blurry, cropped, jpeg artifacts, out of frame, poorly drawn, signature, artist name, missing finger, extra hands, extra foot, nipples hidden, bad composition, poorly rendered genitals"

SPEED_PRESETS = {
    "Ultra Fast (T4)": dict(
        model_quant="Q4", width=480, height=832, frames=8, steps=12, cfg_scale=1.0, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=True
    ),
    "Balanced": dict(
        model_quant="Q6", width=480, height=832, frames=10, steps=18, cfg_scale=1.1, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=False
    ),
    "High Quality": dict(
        model_quant="Q5", width=640, height=1024, frames=10, steps=22, cfg_scale=1.2, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=False
    )
}

def speed_preset_apply(preset):
    params = SPEED_PRESETS.get(preset, SPEED_PRESETS['Ultra Fast (T4)'])
    return (
        params['model_quant'], params['width'], params['height'], params['frames'], params['steps'],
        params['cfg_scale'], params['sampler_name'], params['scheduler'], params['ultra_fast_mode']
    )

with gr.Blocks(title="Wan2.1 GGUF NSFW T2V Generator — Ultra Fast Mode!") as demo:
    gr.Markdown("""# Wan2.1 GGUF NSFW T2V Generator

🚀 <b>Speed Mode:</b> Q4 (fastest), Q5 (best quality), Q6 (middle), plus one-click <span style='color:orange'>Ultra Fast</span> (recommended for T4/free GPUs)! [Run on Lightning AI or RunPod for A100+]
""")

    with gr.Accordion("🚀 Speed Mode: Click a Preset! (Ultra Fast, Balanced, High Quality)", open=True):
        preset_dropdown = gr.Dropdown(list(SPEED_PRESETS.keys()), value="Ultra Fast (T4)", label="Speed Preset", interactive=True)
        model_quant = gr.Dropdown(["Q4", "Q5", "Q6"], label="Model Quantization (Speed/Quality)", value="Q4")
        ultra_fast_mode = gr.Checkbox(label="Force Ultra Fast Mode (T4/preview)", value=True)
        width = gr.Number(label="Width", value=480)
        height = gr.Number(label="Height", value=832)
        frames = gr.Number(label="Frames", value=8)
        steps = gr.Number(label="Steps", value=12)
        cfg_scale = gr.Number(label="CFG Scale", value=1.0)
        sampler_name = gr.Dropdown(["uni_pc", "euler", "dpmpp_2m", "ddim", "lms"], value="uni_pc", label="Sampler")
        scheduler = gr.Dropdown(["simple", "normal", "karras", "exponential"], value="simple", label="Scheduler")

        def _update_by_preset(preset):
            mq, w, h, f, st, cfg, smp, sch, uf = speed_preset_apply(preset)
            return mq, uf, w, h, f, st, cfg, smp, sch
        preset_dropdown.change(_update_by_preset, inputs=preset_dropdown,
                             outputs=[model_quant, ultra_fast_mode, width, height, frames, steps, cfg_scale, sampler_name, scheduler])

    with gr.Tab(label="🎬 Generate"):
        positive_prompt = gr.Textbox(label="Positive Prompt", value=default_positive)
        negative_prompt = gr.Textbox(label="Negative Prompt", value=default_negative)
        lora_name = gr.Textbox(label="LoRA Names (.safetensors, CSV)", placeholder="sexy_butt_v12.safetensors,super_giga.safetensors")
        lora_strength = gr.Textbox(label="LoRA Strengths (CSV/space, matches LoRA count)", value="0.7")
        fps = gr.Number(label="FPS", value=12)
        seed = gr.Number(label="Seed (-1 for random)", value=-1)
        output_format = gr.Dropdown(["mp4", "webm"], label="Output Format", value="mp4")
        run_btn = gr.Button("Generate Video/Image! 🚀")
        output = gr.Video(label="Result", interactive=False)

        def _run(
            pos, neg, mq, w, h, f, st, cfg, smp, sch, loran, loras, s, fpsv, fmt, uf
        ):
            out_path = generate_video(
                positive_prompt=pos,
                negative_prompt=neg,
                width=int(w), height=int(h), seed=int(s), steps=int(st), cfg_scale=float(cfg),
                sampler_name=smp, scheduler=sch, frames=int(f), fps=int(fpsv), lora_name=loran, lora_strength=loras,
                model_quant=mq, output_format=fmt, ultra_fast_mode=uf
            )
            return out_path

        inputs = [
            positive_prompt, negative_prompt, model_quant, width, height, frames, steps, cfg_scale, sampler_name, scheduler,
            lora_name, lora_strength, seed, fps, output_format, ultra_fast_mode
        ]
        run_btn.click(_run, inputs=inputs, outputs=output)

    gr.Markdown("""
⚡ **Platform Tips**:
- For fastest free results, use Q4 + Ultra Fast Mode (default)! Full video usually in ~10s (T4).
- To move to faster GPU: <a href='https://lightning.ai'>Lightning AI</a> (free A100/RTX40), or <a href='https://runpod.io/r/city96-wan21'>RunPod (A100/4090, $0.10/hr)</a>.
- Copy/paste this notebook to these platforms or download and run anywhere with Python & GPU support!
- For notebook copy: Just open the notebook, click "File > Save a copy in Drive" or use the <a href='https://colab.research.google.com/github/[YOUR_NB_LINK].ipynb'>one-click link</a> to Kaibu, Colab, etc.
    """)

demo.queue(api_open=False, default_concurrency_limit=2).launch(share=True)
